# Ckylark で日本語の句構造解析をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_parsing/ckylark.ipynb)

NLP クックブック 14-21 ／ 構文解析

日本語の文を「名詞句」「動詞句」のような句の入れ子構造 (句構造木) に解析したい。日本語と英語の学習済みモデルが付属する PCFG-LA パーザ Ckylark を使う。日本語モデルは KyTea 方式の単語分割を前提にしているので、KyTea も一緒にビルドする。

In [1]:
!apt-get install -y -q autoconf automake libtool libboost-iostreams-dev libboost-program-options-dev > /dev/null
!git clone -q --depth 1 https://github.com/odashi/ckylark
!git clone -q --depth 1 https://github.com/neubig/kytea
!cd ckylark && autoreconf -i > /dev/null 2>&1 && ./configure -q > /dev/null && make -j4 > /dev/null 2>&1
!cd kytea && autoreconf -i > /dev/null 2>&1 && ./configure -q > /dev/null && make -j4 > /dev/null 2>&1 && make install > /dev/null 2>&1 && ldconfig
!pip install -q fugashi unidic-lite
!ls ckylark/data/*.grammar.gz && echo 'This is a pen .' | ckylark/src/bin/ckylark --model ckylark/data/wsj

In [2]:
import subprocess
from nltk import Tree

CKYLARK = "ckylark/src/bin/ckylark"
# 半角英数字・記号を全角にする(日本語モデルの語彙は全角表記)
ZEN = str.maketrans({chr(c): chr(c + 0xFEE0) for c in range(0x21, 0x7F)})

def kytea_segment(sentences):
    text = "\n".join(s.translate(ZEN) for s in sentences) + "\n"
    r = subprocess.run(["kytea", "-notags"], input=text, capture_output=True, text=True)
    return r.stdout.strip().split("\n")

def ckylark_parse(segmented, model="ckylark/data/jdc"):
    # モデル読み込みに数秒かかるので、文はまとめて 1 回で渡す(1 行 1 文、空白区切り)
    r = subprocess.run([CKYLARK, "--model", model], input="\n".join(segmented) + "\n",
                       capture_output=True, text=True)
    return [Tree.fromstring(line)[0] for line in r.stdout.strip().split("\n")]  # 外側の空ラベルを外す

sents = ["太郎は図書館で借りた本を花子に渡した。", "私は昨日友達と映画を見に行きました。"]
seg = kytea_segment(sents)
trees = ckylark_parse(seg)
for s, t in zip(seg, trees):
    print(s)
    print(t)
trees[0].pretty_print()

太郎 は 図書 館 で 借り た 本 を 花子 に 渡 し た 。
(動詞P
  (動詞P
    (助詞はP (名詞 太郎) (助詞は は))
    (動詞P
      (助詞でP (接尾辞P (名詞 図書) (接尾辞 館)) (助詞で で))
      (動詞P
        (助詞をP (名詞P (動詞P (動詞 借り) (助動詞 た)) (名詞 本)) (助詞を を))
        (動詞P
          (助詞にP (名詞 花子) (助詞に に))
          (動詞P (動詞 渡) (語尾P (語尾 し) (助動詞 た)))))))
  (補助記号 。))
私 は 昨日 友達 と 映画 を 見 に 行 き ま し た 。
(動詞P
  (動詞P
    (助詞はP (代名詞 私) (助詞は は))
    (動詞P
      (助詞にP
        (動詞P
          (助詞をP
            (名詞P (助詞とP (名詞P (名詞 昨日) (名詞 友達)) (助詞と と)) (名詞 映画))
            (助詞を を))
          (動詞 見))
        (助詞に に))
      (動詞P (動詞 行) (語尾P (語尾 き) (助動詞P (助動詞 ま) (語尾P (語尾 し) (助動詞 た)))))))
  (補助記号 。))
                                                動詞P                                      
                                         ________|____________________________________    
                                       動詞P                                            |  
      __________________________________|________                                     |   
     |        

In [3]:
# 「名詞句 + 助詞」の句(助詞をP, 助詞にP など)から、助詞の前の部分を項として取り出す
def arguments(tree):
    out = []
    for st in tree.subtrees(lambda t: t.label().startswith("助詞") and t.label().endswith("P")):
        head, particle = st[0], st[-1]
        if isinstance(head, Tree) and head.label() in ("名詞", "名詞P", "代名詞", "接尾辞P"):
            out.append(("".join(head.leaves()), "".join(particle.leaves())))
    return out

for t in trees:
    print(arguments(t))

# 比較: MeCab + UniDic(fugashi)で分割すると、活用語尾の切り方が違うため品詞の推定が崩れる
import fugashi
tagger = fugashi.Tagger()
uni = " ".join(w.surface for w in tagger(sents[1]))
r = subprocess.run([CKYLARK, "--model", "ckylark/data/jdc", "--output-format", "postag"],
                   input=uni + "\n", capture_output=True, text=True)
print(r.stdout.strip())

[('太郎', 'は'), ('図書館', 'で'), ('借りた本', 'を'), ('花子', 'に')]
[('私', 'は'), ('昨日友達と映画', 'を'), ('昨日友達', 'と')]
私/代名詞 は/助詞は 昨日/名詞 友達/名詞 と/助詞と 映画/名詞 を/助詞を 見/動詞 に/助詞に 行き/動詞 まし/形状詞 た/助動詞 。/補助記号


In [4]:
# 英語は WSJ(Penn Treebank)モデル。入力はトークン化済み(句読点も空白で区切る)
en = ckylark_parse(["The quick brown fox jumped over the lazy dog near the river ."],
                   model="ckylark/data/wsj")[0]
en.pretty_print()
print([" ".join(np.leaves()) for np in en.subtrees(lambda t: t.label() == "NP")])

                                 S                                 
       __________________________|_______________________________   
      |                     VP                                   | 
      |                _____|____                                |  
      |               |          PP                              | 
      |               |      ____|________                       |  
      |               |     |             NP                     | 
      |               |     |         ____|________              |  
      |               |     |        |             PP            | 
      |               |     |        |         ____|___          |  
      NP              |     |        NP       |        NP        | 
  ____|__________     |     |     ___|____    |     ___|____     |  
 DT   JJ    JJ   NN  VBD    IN   DT  JJ   NN  IN   DT       NN   . 
 |    |     |    |    |     |    |   |    |   |    |        |    |  
The quick brown fox jumped over the lazy 

- PCFG-LA (潜在注釈付き確率文脈自由文法) は、「NP」などの記号を自動的に細分化した文法で CKY 解析を行う手法で、Berkeley Parser と同じ方式。Ckylark は、未知語があっても解析が失敗しにくいよう工夫されている。
- 日本語モデル (`--model ckylark/data/jdc`) は、KyTea で分割した「日本語単語依存構造コーパス」で学習されており、「渡 し た」「行 き ま し た」のように動詞の語幹と活用語尾を分ける分割を前提にしている。MeCab + UniDic の分割 (「渡し た」) を渡しても解析はできるが、品詞と構造が崩れやすいので、KyTea で分割する。語彙は全角英数字なので、半角は全角に直してから渡す。
- 日本語モデルのラベルは「名詞」「動詞」「語尾」「助動詞」「補助記号」などの品詞と、その句である「名詞P」「動詞P」、さらに助詞ごとの「助詞はP」「助詞をP」「助詞にP」など。英語モデルは Penn Treebank の NP、VP、PP など。出力は 1 文 1 行の S 式で、一番外側にラベルのない括弧が付く (`--add-root-tag` で ROOT にできる)。
- 係り先の誤りは普通に起きる (「太郎は図書館で借りた本を花子に渡した。」で、「図書館で」が「借りた」ではなく「渡した」に係る、など)。N-best 出力はなく、得られるのは最良の 1 つの木だけ。
- モデルの読み込みに日本語で約 2 秒、英語で約 5 秒かかるので、文は 1 回の呼び出しにまとめて渡す。入力中の半角括弧は -LRB- / -RRB- に置き換わる。
- 主なオプション: `--output-format postag` (品詞列だけを出す)、`--add-root-tag`、`--binarize` (二分木で出力)、`--force-generate` (解析に失敗しても平らな木を出す)、`--prune-threshold` (枝刈りの強さ。速度と精度の調整)。
- 2015 年ごろの研究用ツールで、ビルド (Ckylark と KyTea で 2 分ほど) が必要。インストールだけで済ませたい場合や、係り受けも欲しい場合は、前のレシピの Stanza も検討する。